In [1]:
"""
daemon_client.py
----------------
Client-side connector to ssh_daemon.py.

Sends encrypted credentials to the credential daemon over a raw TCP socket
on port 9922, using the same JSON message protocol the daemon expects.

Supports both LAN (direct IP) and WAN (domain/public IP) targets.
The Fernet key must be pre-shared with the daemon — this is what proves
the sender is authorised; the daemon will reject anything it cannot decrypt.

Protocol (newline-terminated JSON):
    Send:    { "type": "credential", "key_hash": "<sha256hex>", "encrypted": "<hex>" }
    Receive: { "status": "ok", "block_index": N, "block_hash": "<hex>" }
             { "status": "error", "reason": "..." }

    Ping:
    Send:    { "type": "ping" }
    Receive: { "status": "ok", "type": "pong" }

    Chain verify:
    Send:    { "type": "verify" }
    Receive: { "status": "ok", "chain_valid": true, "length": N }
"""

import json
import socket
import hashlib
import logging
from pathlib import Path
from cryptography.fernet import Fernet

log = logging.getLogger(__name__)

DEFAULT_PORT    = 9922
DEFAULT_TIMEOUT = 10      # seconds
MAX_RECV_BYTES  = 4096

In [2]:

# ── Fernet ────────────────────────────────────────────────────────────────

def load_fernet(key_path: str = "fernet.key") -> Fernet:
    """
    Load the shared Fernet key.
    Must match the key on the daemon machine — copy it via:
        scp /etc/credential-daemon/fernet.key user@sender:/path/to/fernet.key
    """
    path = Path(key_path)
    if not path.exists():
        raise FileNotFoundError(
            f"Fernet key not found at {path}. "
            "Copy it from the daemon machine: "
            "scp /etc/credential-daemon/fernet.key user@this-machine:./fernet.key"
        )
    return Fernet(path.read_bytes())


In [3]:
import base64
import hashlib

testString = "hello apple"

digest = hashlib.sha256(testString.encode()).digest()  # 32 bytes
key = base64.urlsafe_b64encode(digest)

print(key)

load_fernet("./fernet.key")

b'Jb3wYF0it9Of50enIQ5jvFM5kveiP_m5bkBWJ3kRs0Y='


In [4]:

# ── Low-level socket send ─────────────────────────────────────────────────

def _send_message(host: str, port: int, message: dict, timeout: int = DEFAULT_TIMEOUT) -> dict:
    """Open a TCP connection, send a JSON message, return the parsed response."""
    raw = (json.dumps(message) + "\n").encode()
    try:
        with socket.create_connection((host, port), timeout=timeout) as sock:
            sock.sendall(raw)
            response = sock.recv(MAX_RECV_BYTES)
        return json.loads(response.decode().strip())
    except ConnectionRefusedError:
        raise ConnectionRefusedError(
            f"Daemon not reachable at {host}:{port} — is credential-daemon running?"
        )
    except socket.timeout:
        raise TimeoutError(f"Daemon at {host}:{port} did not respond within {timeout}s")
    except json.JSONDecodeError as e:
        raise ValueError(f"Daemon returned invalid JSON: {e}")



In [5]:
import socket
import threading
import json

HOST = "127.0.0.1"
PORT = 9200

# --- fake daemon server ---
def fake_server():
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.bind((HOST, PORT))
    srv.listen(1)

    conn, _ = srv.accept()
    data = conn.recv(4096)

    # simulate real daemon response
    response = {
        "status": "ok",
        "block_index": 7,
        "block_hash": "testhash123"
    }

    conn.sendall((json.dumps(response) + "\n").encode())
    conn.close()
    srv.close()

# start server
threading.Thread(target=fake_server, daemon=True).start()

# --- test client ---
msg = {"type": "credential"}

resp = _send_message(HOST, PORT, msg)

assert resp["status"] == "ok"
assert resp["block_index"] == 7

print("Integration test passed:", resp)

Integration test passed: {'status': 'ok', 'block_index': 7, 'block_hash': 'testhash123'}


In [6]:

# ── Public API ────────────────────────────────────────────────────────────

def ping(host: str, port: int = DEFAULT_PORT) -> bool:
    """
    Check if the daemon is alive.
    Returns True if it responds with pong, False otherwise.
    """
    try:
        resp = _send_message(host, port, {"type": "ping"})
        alive = resp.get("type") == "pong"
        log.info(f"Ping {host}:{port} → {'alive' if alive else 'no pong'}")
        return alive
    except Exception as e:
        log.warning(f"Ping failed: {e}")
        return False



In [7]:
import socket
import threading
import json
import time

# --- fake daemon that replies to ping ---
def fake_daemon(host, port):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((host, port))
    srv.listen(1)

    conn, _ = srv.accept()
    data = conn.recv(4096)

    msg = json.loads(data.decode().strip())

    if msg.get("type") == "ping":
        response = {"type": "pong"}
    else:
        response = {"type": "unknown"}

    conn.sendall((json.dumps(response) + "\n").encode())
    conn.close()
    srv.close()


# --- start fake server ---
HOST = "127.0.0.1"
PORT = 9400

threading.Thread(target=fake_daemon, args=(HOST, PORT), daemon=True).start()

time.sleep(0.5)  # let server start

# --- run your API ---
print("PING RESULT:", ping(HOST, PORT))

PING RESULT: True


In [8]:

def verify_chain(host: str, port: int = DEFAULT_PORT) -> dict:
    """
    Ask the daemon to verify its ledger chain integrity.
    Returns { chain_valid: bool, length: int }
    """
    resp = _send_message(host, port, {"type": "verify"})
    if resp.get("status") != "ok":
        raise RuntimeError(f"Chain verify failed: {resp.get('reason')}")
    log.info(f"Chain @ {host}: valid={resp['chain_valid']}, length={resp['length']}")
    return resp


In [9]:
import socket
import threading
import json
import time

def fake_verify_daemon(host, port):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((host, port))
    srv.listen(1)

    conn, _ = srv.accept()
    data = conn.recv(4096)

    msg = json.loads(data.decode().strip())

    if msg.get("type") == "verify":
        response = {
            "status": "ok",
            "chain_valid": True,
            "length": 5
        }
    else:
        response = {
            "status": "error",
            "reason": "invalid request"
        }

    conn.sendall((json.dumps(response) + "\n").encode())
    conn.close()
    srv.close()

In [10]:
HOST = "127.0.0.1"
PORT = 9500

# start fake daemon
threading.Thread(
    target=fake_verify_daemon,
    args=(HOST, PORT),
    daemon=True
).start()

time.sleep(0.5)

# call your API
resp = verify_chain(HOST, PORT)

print("VERIFY RESPONSE:", resp)

assert resp["chain_valid"] is True
assert resp["length"] == 5

print("Test passed ✔")

VERIFY RESPONSE: {'status': 'ok', 'chain_valid': True, 'length': 5}
Test passed ✔


In [11]:

def send_to_daemon(
    credential: dict,
    host: str,
    fernet: Fernet,
    port: int = DEFAULT_PORT,
    timeout: int = DEFAULT_TIMEOUT,
) -> dict:
    """
    Encrypt a credential and deliver it to the daemon for storage.

    Args:
        credential: dict with at least a 'key' field
        host:       daemon machine IP or hostname (LAN or WAN)
        fernet:     shared Fernet instance (key must match daemon's key)
        port:       daemon port (default 9922)
        timeout:    socket timeout in seconds

    Returns:
        { "status": "ok", "block_index": N, "block_hash": "<hex>" }

    Raises:
        ValueError, ConnectionRefusedError, TimeoutError, RuntimeError
    """
    key_value = credential.get("key")
    if not key_value:
        raise ValueError("Credential missing 'key' field")

    # Encrypt
    plaintext = key_value.encode()
    encrypted = fernet.encrypt(plaintext)
    key_hash  = hashlib.sha256(plaintext).hexdigest()

    message = {
        "type":      "credential",
        "key_hash":  key_hash,
        "encrypted": encrypted.hex(),
    }

    log.info(f"Sending credential to daemon @ {host}:{port} ...")
    resp = _send_message(host, port, message, timeout=timeout)

    if resp.get("status") != "ok":
        raise RuntimeError(f"Daemon rejected credential: {resp.get('reason')}")

    log.info(
        f"Credential stored — block #{resp['block_index']}, "
        f"hash: {resp['block_hash'][:16]}..."
    )
    return resp
    

In [12]:
import socket
import threading
import json
import hashlib
import time
from cryptography.fernet import Fernet

def fake_daemon(host, port):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((host, port))
    srv.listen(1)

    conn, _ = srv.accept()
    data = conn.recv(8192)

    msg = json.loads(data.decode().strip())

    if msg.get("type") == "credential":
        response = {
            "status": "ok",
            "block_index": 42,
            "block_hash": hashlib.sha256(b"test-block").hexdigest()
        }
    else:
        response = {
            "status": "error",
            "reason": "invalid message type"
        }

    conn.sendall((json.dumps(response) + "\n").encode())
    conn.close()
    srv.close()

In [13]:
HOST = "127.0.0.1"
PORT = 9600

# shared key (must match daemon expectation in real system)
fernet = Fernet(Fernet.generate_key())

# start fake daemon
threading.Thread(
    target=fake_daemon,
    args=(HOST, PORT),
    daemon=True
).start()

time.sleep(0.5)

# --- input credential ---
credential = {"key": "my-secret-value"}

# --- call function ---
resp = send_to_daemon(
    credential=credential,
    host=HOST,
    port=PORT,
    fernet=fernet,
)

print("RESPONSE:", resp)

# --- assertions ---
assert resp["status"] == "ok"
assert "block_index" in resp
assert "block_hash" in resp

print("Test passed ✔")

RESPONSE: {'status': 'ok', 'block_index': 42, 'block_hash': '4c655693123a7d01489f4c3f5c93f02c95682c9f9c6b3c385068bb192e0ee58c'}
Test passed ✔
